In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

dataset = dataset.map(
    lambda x: {
        "combined_text": x["prompt"] + " " + x["A"]
    }
)

print(len(dataset[51]["combined_text"]))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]


tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

prompts = list(dataset["prompt"])

encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


In [8]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("bert-base-uncased")

print(config.hidden_size)          
print(config.num_attention_heads)  

print(config.hidden_size // config.num_attention_heads)

768
12
64


In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Tokenize the prompt from row 0 (default settings)
inputs = tokenizer(
    dataset[0]["prompt"],
    return_tensors="pt"
)

# Forward pass
outputs = model(**inputs)

# Print the shape
print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [10]:
outputs = model(**inputs)

In [11]:
# CLS embedding (first token)
cls_embedding = outputs.last_hidden_state[0, 0]

# Sum of first 5 values
answer = cls_embedding[:5].sum().item()

print(answer)
print(round(answer, 4))

-1.200096845626831
-1.2001


In [12]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Load model with attentions
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Show token indices
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for i, token in enumerate(tokens):
    print(i, token)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]


In [15]:
fusion_index = tokens.index("fusion")
print(fusion_index)

4


In [16]:
# Last layer, first head
attention = outputs.attentions[-1][0, 0]

# CLS (token 0) -> fusion
weight = attention[0, fusion_index].item()

print(weight)
print(round(weight, 4))

0.10247313976287842
0.1025


In [17]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

# Load MiniLM model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Row 0
prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(
    prompt_embedding,
    option_b_embedding
)

print(similarity)
print(round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

tensor([[0.7658]], device='cuda:0')
0.7658


In [18]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import numpy as np

# -----------------------
# Load dataset
# -----------------------
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

# -----------------------
# MAP@3
# -----------------------
def map3(actual, predictions):
    score = 0
    for a, preds in zip(actual, predictions):
        if a in preds:
            score += 1 / (preds.index(a) + 1)
    return score / len(actual)

# -----------------------
# TF-IDF Pipeline
# -----------------------
tfidf_predictions = []

for row in dataset:

    docs = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(docs)

    sims = cosine_similarity(X[0:1], X[1:]).flatten()

    labels = np.array(["A","B","C","D","E"])

    top3 = labels[np.argsort(sims)[::-1][:3]]

    tfidf_predictions.append(list(top3))

# -----------------------
# MiniLM Pipeline
# -----------------------
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

minilm_predictions = []

for row in dataset:

    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    option_embeddings = model.encode(
        [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ],
        convert_to_tensor=True
    )

    sims = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0].cpu().numpy()

    labels = np.array(["A","B","C","D","E"])

    top3 = labels[np.argsort(sims)[::-1][:3]]

    minilm_predictions.append(list(top3))

# -----------------------
# MAP@3
# -----------------------
actual = [row["answer"] for row in dataset]

minilm_map3 = map3(actual, minilm_predictions)

print("MiniLM MAP@3 =", minilm_map3)

# -----------------------
# Improvement count
# -----------------------
count = 0

for gt, tfidf_pred, minilm_pred in zip(
    actual,
    tfidf_predictions,
    minilm_predictions
):

    if gt not in tfidf_pred and gt in minilm_pred:
        count += 1

print("Improved questions =", count)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM MAP@3 = 0.42308333333333487
Improved questions = 564


In [19]:
from datasets import load_dataset
from transformers import pipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

# Zero-shot classifier (defaults to facebook/bart-large-mnli)
classifier = pipeline("zero-shot-classification")

# Row with index 1 (2nd row)
row = dataset[1]

sequence = row["prompt"]

candidate_labels = [
    row["A"],
    row["B"],
    row["C"]
]

result = classifier(
    sequence,
    candidate_labels
)

print(result)
print("Top label:", result["labels"][0])
print("Top score:", result["scores"][0])
print("Rounded:", round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [20]:
from datasets import load_dataset
from transformers import pipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

classifier = pipeline("zero-shot-classification")

row = dataset[1]

sequence = row["prompt"]

candidate_labels = [
    row["A"],
    row["B"],
    row["C"]
]

# Previous question (Softmax)
result_softmax = classifier(
    sequence,
    candidate_labels
)

# Current question (Sigmoid)
result_sigmoid = classifier(
    sequence,
    candidate_labels,
    multi_label=True
)

sum_softmax = sum(result_softmax["scores"])
sum_sigmoid = sum(result_sigmoid["scores"])

print("Softmax Sum :", sum_softmax)
print("Sigmoid Sum :", sum_sigmoid)
print("Absolute Difference :", abs(sum_softmax - sum_sigmoid))
print("Rounded :", round(abs(sum_softmax - sum_sigmoid), 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Softmax Sum : 0.9999999701976776
Sigmoid Sum : 0.0005095975611766335
Absolute Difference : 0.999490372636501
Rounded : 0.9995


In [23]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row = dataset[0]

prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


B
